<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-09-multimodal-and-pretrained/lesson-9.5-vision-models/notebooks/GCP_Capstone_9.5_VisionModels.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9.5 How Vision Models See — Patches, Attention, Contrast, and the Corpus's Real Figures
**Netsetos GenAI Engineering — GCP Capstone** · Module 9 · rebuilt on the live lane, 9 September 2026

The theory that makes Module 9 work, unchanged where it was right: an image is a sequence of patches (which is why you have paid per tile since 2.1), attention can be reshaped back onto the image, contrastive training puts pictures and words in one space, RRF fuses two rankings, diffusion runs in two directions, the OSS lane has a licence question. What changed is the evidence: the cross-modal cells now embed the corpus's three real figures with `gemini-embedding-2-preview`, and the gate compares that ranking with the lane's own caption route on the same figures. The `media_chunks` index stays a printed proposal, and the gate is the reason.


## Setup
Two clients: `gen` on global for generation, `emb` on us-central1 for embeddings - the endpoint rule, both halves.


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 Pillow==12.3.0 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson

# EMBEDDINGS ARE REGIONAL - the other half of the endpoint rule (9.5, 9.6): this client is for
# embed_content only; everything generative goes through `gen`, on global.
emb = genai.Client(enterprise=True, project=PROJECT_ID, location=REGION)
import numpy as np                                 # Colab's own numpy; nothing here needs a pin

print("kit:", KIT, "| API:", API_URL, "| media:", f"gs://{MEDIA_BUCKET}")


## Cell 1: The corpus's media


In [ ]:
import json, requests, time
from google.cloud import storage

# THE ROUTES 9.4's Studio stands on, called the way the UI calls them: one ID token per request,
# minted AS the roster member, audience = the API (7.3's hour-long fuse never arms). The body names
# the tenant; the API checks the caller's email on that tenant's roster before it spends a paisa.
def api(path: str, body: dict | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route as documind-ui-sa. Returns (status, json-or-text) - never raises on 4xx,
    because a refusal is data this module reads (the outsider cells)."""
    r = requests.post(f"{API_URL}{path}", json=body,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

# The corpus's media objects, read as YOU (the Colab credential is a project owner; the lane's
# services read them as their own accounts). One client, both buckets.
gcs = storage.Client(project=PROJECT_ID)

def media_objects(prefix: str = f"{TENANT}/") -> list[str]:
    """Every image, video or recording under the tenant's prefix of the uploads bucket."""
    return sorted(b.name for b in gcs.list_blobs(UPLOAD_BUCKET, prefix=prefix)
                  if b.name.lower().endswith((".png", ".jpg", ".jpeg", ".mp4", ".mp3")))

def gcs_bytes(uri: str) -> bytes:
    bucket, _, name = uri.removeprefix("gs://").partition("/")
    return gcs.bucket(bucket).blob(name).download_as_bytes()

FIG3  = f"gs://{UPLOAD_BUCKET}/{TENANT}/annual_report_2026_fig3.png"
INV   = f"gs://{UPLOAD_BUCKET}/{TENANT}/inv_2026_0412.png"
PAGE  = f"gs://{UPLOAD_BUCKET}/{TENANT}/payment_of_bonus_act_1965_p30.png"
VIDEO = f"gs://{UPLOAD_BUCKET}/{TENANT}/townhall_2026_q1.mp4"
POSH  = f"gs://{UPLOAD_BUCKET}/{TENANT}/posh_act_2013.pdf"

have = media_objects()
HAS_VIDEO = f"{TENANT}/townhall_2026_q1.mp4" in have
print("media in the corpus:", have or "NONE - run `make media` and `make ingest-corpus` from deploy/ (README: media is a document)")
print("video:", "present" if HAS_VIDEO else "absent (make media MEDIA_ARGS=--video, or drop a recording in) - the video cells will say so")
assert have, "no media under the tenant's prefix: nothing in this lesson can cite a figure until the corpus holds one"


## Cell 2: You have been paying for patches since Module 2


In [ ]:
# You have been paying for patches since Module 2.
#
# 2.1 measured it: an image <=384x384 costs 258 tokens, and anything larger is
# cut into 768x768 TILES at 258 tokens each - so a 1024x1024 image is ~1,290.
# That is not a billing quirk. It is the model turning a picture into a
# SEQUENCE, because a transformer cannot see a grid; it can only read a list.
def patchify(h: int, w: int, patch: int = 16) -> tuple[int, int, int]:
    """How many patches a h x w image becomes at patch x patch each."""
    gh, gw = h // patch, w // patch
    return gh, gw, gh * gw


for size in (224, 384, 768, 1024):
    gh, gw, n = patchify(size, size)
    print(f'{size:>5}x{size:<5} -> {gh:>2} x {gw:<2} grid = {n:>4} patches')

print()
# ViT-B/16 on the classic 224x224 input: 14 x 14 = 196 patches, plus one [CLS]
# token the model uses as the image's summary vector.
gh, gw, n = patchify(224, 224, 16)
print(f'ViT-B/16 at 224: {n} patches + 1 [CLS] = {n + 1} tokens in the sequence')

# What Gemini actually charges, as MEASURED in 2.1 and 9.1:
MEASURED = {
    '<=384x384 (flat rate)': 258,
    'one 768x768 tile':      258,
    '1024x1024':            1290,     # ~5 tiles, not the 4 a naive ceil() predicts
    'one PDF page':          258,     # rendered as an image
}
print()
for label, tokens in MEASURED.items():
    print(f'  {label:24} {tokens:>5} tokens')

print()
print('  1024x1024 is ~1,290 = 5 x 258, not the 4 tiles ceiling division predicts.')
print('  Which is exactly why 2.1 says MEASURE the image token cost rather than')
print('  deriving it: the tiling has a step your arithmetic does not.')
print()
print("  count_tokens(contents=[part]) is the answer, every time.")


## Cell 3: Where did it look?


In [ ]:
# What a patch sequence lets you ask: WHERE did the model look?
#
# Attention is a matrix over the sequence. Reshape one head's attention from the
# [CLS] token back onto the patch grid and you get a heat map in image space -
# the same trick that turns "the model said invoice" into "the model was looking
# at the total row when it said invoice".
def cls_attention_map(attn: np.ndarray, grid: int) -> np.ndarray:
    """attn: (tokens,) attention FROM [CLS] TO every token. Returns grid x grid.

    Token 0 is [CLS] attending to itself - drop it, or one cell dominates the
    map and every heat map you produce looks like a bullseye at the origin.
    """
    patches = attn[1:]
    assert patches.size == grid * grid, (
        f'{patches.size} patch attentions do not fill a {grid}x{grid} grid')
    m = patches.reshape(grid, grid)
    return (m - m.min()) / (np.ptp(m) + 1e-9)          # normalise for display


# A synthetic head that attends to the centre, so the map is checkable.
g = 14
yy, xx = np.mgrid[0:g, 0:g]
centre = np.exp(-((yy - 6.5) ** 2 + (xx - 6.5) ** 2) / 8.0)
fake = np.concatenate([[0.9], centre.ravel()])       # [CLS] first

m = cls_attention_map(fake, g)
print('attention map', m.shape, 'range', round(float(m.min()), 3),
      '-', round(float(m.max()), 3))
peak = np.unravel_index(m.argmax(), m.shape)
print('peak at patch', peak, '-> pixel box',
      (peak[0] * 16, peak[1] * 16, peak[0] * 16 + 16, peak[1] * 16 + 16))


In [ ]:
# The failure the assertion exists for, made explicit.
full = np.concatenate([[0.9], centre.ravel()])          # 197 values: [CLS] + 196
try:
    cls_attention_map(np.concatenate([full, [0.0]]), 14)   # 198 -> caught
except AssertionError as e:
    print('caught:', e)

# But this one is NOT caught by any shape check, because 196 fits 14x14
# perfectly - it is simply the wrong 196 values, shifted by one.
wrong = full[:-1]           # kept [CLS], dropped the last patch
m_wrong = cls_attention_map(np.concatenate([[0.0], wrong]), 14)
m_right = cls_attention_map(full, 14)
import numpy as _np
print('peak, correct slice:', _np.unravel_index(m_right.argmax(), m_right.shape))
print('peak, off-by-one   :', _np.unravel_index(m_wrong.argmax(), m_wrong.shape))
print()
print('Both are valid 14x14 maps. Only one is the truth - which is why the')
print('shape assertion is necessary but NOT sufficient, and why you sanity-check')
print('a heat map against an image you already understand.')


## Cell 4: The contrastive objective - CLIP vs SigLIP


In [ ]:
# The contrastive objective, which is the whole idea behind cross-modal search.
#
# Take N (image, caption) pairs. Embed both sides. Now push the N matching pairs
# together and the N^2 - N mismatched pairs apart. Do that over enough data and
# "a photo of a whiteboard" and an actual whiteboard photo land in the same
# neighbourhood - WITHOUT anyone ever labelling a whiteboard class.
def clip_loss(img: np.ndarray, txt: np.ndarray, temp: float = 0.07) -> float:
    """CLIP: softmax over the batch. Each row competes with the whole batch."""
    img = img / np.linalg.norm(img, axis=1, keepdims=True)
    txt = txt / np.linalg.norm(txt, axis=1, keepdims=True)
    logits = img @ txt.T / temp
    # cross-entropy against the diagonal, both directions
    def ce(x):
        x = x - x.max(axis=1, keepdims=True)
        return float(np.mean(-np.diag(x) + np.log(np.exp(x).sum(axis=1))))
    return (ce(logits) + ce(logits.T)) / 2


def siglip_loss(img: np.ndarray, txt: np.ndarray,
                temp: float = 0.07, bias: float = -10.0) -> float:
    """SigLIP: a SIGMOID per pair. No softmax, so no competition across the batch.

    That single change is why SigLIP scales: CLIP's softmax needs every pair in
    the batch normalised together, so the batch is a global object and doubling
    it doubles the coordination. Sigmoid treats each pair independently, so the
    batch is just data.
    """
    img = img / np.linalg.norm(img, axis=1, keepdims=True)
    txt = txt / np.linalg.norm(txt, axis=1, keepdims=True)
    logits = img @ txt.T / temp + bias
    labels = 2 * np.eye(len(img)) - 1                # +1 matched, -1 mismatched
    return float(np.mean(np.log1p(np.exp(-labels * logits))))


rng = np.random.default_rng(0)
d, n = 32, 8
paired = rng.normal(size=(n, d))
img = paired + rng.normal(scale=0.45, size=(n, d))   # matched, but genuinely noisy
txt = paired + rng.normal(scale=0.45, size=(n, d))
shuffled = txt[rng.permutation(n)]                   # deliberately mismatched

for name, fn in (('CLIP (softmax)', clip_loss), ('SigLIP (sigmoid)', siglip_loss)):
    good, bad = fn(img, txt), fn(img, shuffled)
    print(f'{name:20} aligned {good:6.3f}   shuffled {bad:6.3f}   '
          f'{"separates" if bad > good else "FAILS TO SEPARATE"}')


## Cell 5: Cross-modal search on Vertex, on the corpus's real figures
`gemini-embedding-2-preview` at 768-d, regional. Preview, not GA - read the comment before you build on it. The cell degrades to the caption route if the model is not available to the project today.


In [ ]:
# CROSS-MODAL SEARCH ON VERTEX, ON THE CORPUS'S REAL FIGURES. gemini-embedding-2-preview is Google's first
# natively multimodal embedding model: text, images, video, audio and PDFs in ONE vector space, which is the
# only reason a text query can retrieve a picture at all. VERIFIED 2026-09-04: the ID carries -preview, and
# it is Public Preview, not GA - read the box before you build an index on it. Embeddings are REGIONAL: the
# `emb` client is us-central1. The GA path for image+text in one space is multimodalembedding@001 at 1408-d;
# keep the 768-d schema and the swap is a re-embed, not a re-architecture.
EMBED_MODEL = "gemini-embedding-2-preview"
DIM = 768                # near-peak quality at a quarter of 3072's storage; Matryoshka: truncate, no retrain

def embed_text(text: str) -> list[float]:
    r = emb.models.embed_content(model=EMBED_MODEL, contents=text,
                                 config=types.EmbedContentConfig(output_dimensionality=DIM))
    return r.embeddings[0].values

def embed_image(uri: str) -> list[float]:
    part = types.Part.from_bytes(data=gcs_bytes(uri), mime_type="image/png")
    r = emb.models.embed_content(model=EMBED_MODEL, contents=part,
                                 config=types.EmbedContentConfig(output_dimensionality=DIM))
    return r.embeddings[0].values

IMAGES = {"annual_report_2026_fig3.png": FIG3, "inv_2026_0412.png": INV, "payment_of_bonus_act_1965_p30.png": PAGE}
try:
    INDEX = {name: embed_image(uri) for name, uri in IMAGES.items()}
    CROSS_MODAL_OK = True
    print(f"{len(INDEX)} real images embedded at {DIM}-d with {EMBED_MODEL} (regional)")
except Exception as e:
    CROSS_MODAL_OK = False
    print(f"{EMBED_MODEL} is not available to this project/region today ({type(e).__name__}: {str(e)[:120]}).")
    print("Preview models move; the gate below then runs on the caption route alone, which is the lane's.")


## Cell 6: The gate - a query finds the same figure both ways
The cross-modal ranking of the three real images against the lane's caption route, on three questions. Then an unrelated query, so you see what *no good match* looks like.


In [ ]:
# THE GATE: a text query and the caption route must find the SAME figure. Two rankings over the corpus's
# three real images - the cross-modal one (pixels embedded, query embedded, cosine) and the lane's own
# (retrieve(): the worker's caption, matched by text). The lane does not run the first one (D3: captions
# are the retrievable body; a Preview model stays out of the product), and this cell is why that is a
# defensible choice: on this corpus the caption finds the figure a cross-modal index would find.
def cosine(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

QUERIES = {"revenue by region, FY2025 against FY2026, which region fell": "annual_report_2026_fig3.png",
           "an invoice with a GST line and a total payable": "inv_2026_0412.png",
           "a schedule showing set on and set off of bonus year by year": "payment_of_bonus_act_1965_p30.png"}

agree = 0
for q, expect in QUERIES.items():
    hits = documind_tools.retrieve(q, tenant_id=TENANT, top_k=8, brain="direct")
    figs = [c["source_uri"].rsplit("/", 1)[-1] for c in hits.get("citations", []) if c.get("kind") == "figure"]
    caption_top = figs[0] if figs else None
    if CROSS_MODAL_OK:
        ranked = sorted(((cosine(embed_text(q), v), k) for k, v in INDEX.items()), reverse=True)
        cross_top = ranked[0][1]
        print(f"{q[:52]:52} caption-> {str(caption_top):34} cross-modal-> {cross_top:34} ({ranked[0][0]:.3f})")
        agree += (caption_top == cross_top == expect)
    else:
        print(f"{q[:52]:52} caption-> {caption_top}")
        agree += (caption_top == expect)
    assert caption_top == expect, f"the caption route ranked {caption_top} first for {q!r}; expected {expect} - is the media ingested?"
print(f"\n{agree}/{len(QUERIES)} queries: the routes agree on the figure, and it is the right one")

if CROSS_MODAL_OK:
    # And break it, because a search that has never missed proves nothing: an unrelated query scores low
    # everywhere - the signal a threshold turns into answerable=False.
    unrelated = "a photograph of a mountain lake at dawn"
    scores = sorted(((cosine(embed_text(unrelated), v), k) for k, v in INDEX.items()), reverse=True)
    print(f"unrelated query -> best {scores[0][1]} at {scores[0][0]:.3f}: THAT is what 'no good match' looks like")


## Cell 7: The `media_chunks` index, as a proposal
Equality filter first, vector field last. Printed, not written: the lane's one index holds the captions, and the gate above is why.


In [ ]:
# THE media_chunks INDEX, AS A PROPOSAL - printed, not written. It copies firestore_indexes.tf exactly:
# EQUALITY FILTER FIRST, VECTOR FIELD LAST; Firestore refuses the reverse and names an index that looks
# correct. The lane does NOT adopt it (Module 9 plan, D3): a second collection on a Preview model is a
# second retrieve(), and the gate above showed the caption route finding the same figures on this corpus.
# When it is adopted, this is the shape, and RRF (next cell) is how its ranking joins the caption ranking.
TF = """resource "google_firestore_index" "media_chunks_vector" {
  project     = var.project_id
  database    = google_firestore_database.main.name
  collection  = "media_chunks"
  query_scope = "COLLECTION"

  # tenant_id first. Always. A media index without it will happily return
  # another customer's whiteboard photo to a text query.
  fields {
    field_path = "tenant_id"
    order      = "ASCENDING"
  }
  fields {
    field_path = "__name__"
    order      = "ASCENDING"
  }
  fields {
    field_path = "embedding"
    vector_config {
      dimension = 768          # gemini-embedding-2-preview truncated to 768
      flat {}
    }
  }
}
"""
print(TF)
print("Not written to the kit. The lane's one index is chunks_vector (tenant_id, __name__, embedding) on text-embedding-005,")
print("and a figure's caption is a chunk in it - one index, one retrieve(), no Preview model in the product.")


## Cell 8: Fusing two rankings with RRF


In [ ]:
# Fusing two rankings: RRF.
#
# You now have two retrievers over the same corpus - dense vectors and whatever
# keyword search you already had. Their SCORES are not comparable (one is a
# cosine, one is a BM25), so you cannot average them. RRF ignores the scores
# and uses only the RANKS, which is exactly why it works without tuning.
def rrf(rankings: list[list[str]], k: int = 60) -> list[tuple[str, float]]:
    """Reciprocal Rank Fusion. k=60 is the value the original paper used."""
    scores: dict[str, float] = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda kv: -kv[1])


dense = ['fig_arch', 'fig_costs', 'txt_policy', 'fig_gantt']
sparse = ['txt_policy', 'fig_arch', 'txt_msa', 'fig_costs']

print('dense :', dense)
print('sparse:', sparse)
print()
for doc, score in rrf([dense, sparse]):
    where = []
    if doc in dense:
        where.append(f'dense#{dense.index(doc) + 1}')
    if doc in sparse:
        where.append(f'sparse#{sparse.index(doc) + 1}')
    print(f'  {doc:12} {score:.5f}   {" + ".join(where)}')

print()
print('fig_arch wins: 2nd and 1st beats a single 1st, because agreement across')
print('two independent retrievers is worth more than one confident retriever.')


## Cell 9: Diffusion, forward and reverse


In [ ]:
# Diffusion, in the only two directions it has.
#
# FORWARD is not learned: add gaussian noise on a schedule until the image is
# indistinguishable from static. REVERSE is the model: predict the noise that
# was added at step t, subtract it, repeat. Generation is the reverse walk.
def forward_noise(x0: np.ndarray, t: int, T: int = 1000) -> np.ndarray:
    """One closed-form jump to step t. No loop - that is the trick."""
    betas = np.linspace(1e-4, 0.02, T)
    alpha_bar = np.cumprod(1.0 - betas)[t]
    noise = np.random.default_rng(0).normal(size=x0.shape)
    return np.sqrt(alpha_bar) * x0 + np.sqrt(1 - alpha_bar) * noise


signal = np.sin(np.linspace(0, 6.28, 64))   # not constant: corrcoef needs variance
print(f'{"step":>6}  {"signal kept":>12}  {"looks like":<20}')
for t in (0, 100, 400, 800, 999):
    x = forward_noise(signal, t)
    kept = float(np.corrcoef(x, signal)[0, 1]) if t else 1.0
    print(f'{t:>6}  {kept:>12.3f}  {"the image" if kept > 0.7 else "mostly noise" if kept > 0.2 else "static":<20}')

print()
# CLASSIFIER-FREE GUIDANCE is one line of arithmetic and the dial everyone turns:
#   eps = eps_uncond + scale * (eps_cond - eps_uncond)
# scale 1.0 ignores the prompt's pull entirely; 7-8 is the usual range; push it
# to 20 and you get a saturated, over-literal image because you have amplified
# the difference between "with prompt" and "without" far past what the model
# actually predicted.
for scale in (1.0, 3.0, 7.5, 20.0):
    uncond, cond = 0.20, 0.35
    print(f'  cfg {scale:>4}  ->  eps {uncond + scale * (cond - uncond):.3f}'
          f'{"   (prompt ignored)" if scale == 1.0 else "   (over-driven)" if scale >= 20 else ""}')


## Cell 10: The OSS lane and the licence question


In [ ]:
# The OSS lane, and why this lesson does not paste benchmark numbers.
#
# open_clip (ViT-B-32), DINOv2 and FLUX.1-schnell all run on a free Colab T4.
# They are worth running once so the words above stop being words:
#
#   open_clip ViT-B-32   zero-shot classification with NO training. You hand it
#                        candidate captions and it ranks them - the contrastive
#                        objective from step 4, used directly.
#   DINOv2               self-supervised, no captions at all. Freeze it, fit a
#                        linear probe on top, and you get a document-type
#                        classifier from a few hundred examples - which is what
#                        replaces the hand-built doc_type feature from 5.5.
#   FLUX.1-schnell       few-step diffusion. Watch the step count change the
#                        image and the guidance scale change how literally the
#                        prompt is obeyed.
#
# LICENCES: check the model card before anything ships. These terms change more
# often than model IDs, and "it was Apache when I read the blog post" is not a
# defence. What to look for, every time:
#
#   1. Commercial use permitted?           (some weights are research-only)
#   2. Output ownership and restrictions?  (some licences constrain the IMAGES)
#   3. Redistribution of weights?          (matters the moment you bake a
#                                           container and push it to Artifact
#                                           Registry - that is redistribution)
#
# For DocuMind the third one bites first: a Cloud Run image with weights baked
# in is a copy you are distributing to your own infrastructure, and some
# licences care about that even when nobody outside your project can pull it.
print('OSS runs are cached in the repo - see the exercise grid.')


## Where this goes
- **9.6** is the lane's answer to everything above: the caption is the quote, the figure rides beside it, and the one `retrieve()` returns both.
- The cross-modal index is the exercise grid's Challenge, not the product - until a GA model and a measured corpus say otherwise.

## ✅ Lesson 9.5 complete
- ✅ Patches and tiles: why an image is a sequence and what it costs
- ✅ An attention map reshaped onto the image, and the off-by-one no shape check catches
- ✅ CLIP versus SigLIP, and why the sigmoid scales
- ✅ The corpus's three real figures embedded cross-modally; the gate against the lane's caption route
- ✅ The `media_chunks` index as a proposal, with the reason the lane does not adopt it
- ✅ RRF, diffusion in two directions, the licence question
